In [1]:
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint,HuggingFaceEmbeddings
from langchain_community import document_loaders
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma.vectorstores import Chroma
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

c:\Users\babuh\OneDrive\Documents\RAG_Application\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

False

In [2]:
llm_endpoint = HuggingFaceEndpoint(
    repo_id='meta-llama/Llama-3.1-8B-Instruct',
    task='text-generation',
    temperature=0.3,
    max_new_tokens=512,
)

llm = ChatHuggingFace(llm=llm_endpoint)

NameError: name 'HuggingFaceEndpoint' is not defined

In [23]:
chat_endpoint = HuggingFaceEndpoint(
    repo_id='meta-llama/Llama-3.1-8B-Instruct',
    task='text-generation',
    temperature=0.7,
    max_new_tokens=512,
)

Model = ChatHuggingFace(llm=chat_endpoint)

In [ ]:
Template = ChatPromptTemplate.from_messages([
    ("system","""You are a professional AI assistant.
        Respond in:
        Clear professional English
        Grammatically correct sentences
        Concise but informative explanations
        Structured formatting when necessary
        Formal technical tone
        If context is insufficient, clearly state that the information is unavailable."""
    ),
    ("human","Question: {query} Context:{context}")
])

In [25]:
EmbeddingModel = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
VectorStore = Chroma(
    embedding_function=EmbeddingModel,
    collection_name='sample',
    persist_directory='db'
)

In [27]:
documents = document_loaders.PyPDFLoader(file_path='Personal_Biodata_Karthik.pdf')
data = documents.load()

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', ' ', '']
)
split_data = splitter.split_documents(data)
print(f"Total chunks: {len(split_data)}")

Total chunks: 3


In [29]:
VectorStore.add_documents(split_data)

['464b5d6c-2264-4a11-bfbd-d9544f52fca8',
 '4ae8bd7d-2c5c-4ca3-bae8-27442a58c63a',
 '8597b119-9474-4e34-b35f-35ccf8cff058']

In [30]:
retriever = MultiQueryRetriever.from_llm(
    retriever=VectorStore.as_retriever(search_kwargs={"k": 3}),
    llm=llm
)

In [31]:
chain = (
    {"context": retriever, "query": RunnablePassthrough()}
    | Template
    | Model
    | StrOutputParser()
)

In [32]:
Response = chain.invoke('Who is Karthik,Explain what you know about Karthik')

In [33]:
Response

'**Information about Karthik**\n\nBased on the provided documents, here\'s what I\'ve gathered about Karthik:\n\n### Name and Address\n\n* Full Name: Nagari Karthik Reddy\n* Address: Alathur (Village), Karvetinagaram, Chittoor District\n\n### Family Details\n\n* Father\'s Name: N. Jayachandra Reddy\n* Mother\'s Name: N. Revathi\n* Brother\'s Name: N. Suresh (Chartered Accountant, KPMG)\n* Sister-in-Law\'s Name: N. Srilekha (Chartered Accountant)\n\n### Education and Career\n\n* Undergraduate: B.Tech from National Institute of Technology, Goa\n* Postgraduate: M.Tech from National Institute of Technology, Calicut\n* Designation: Software Developer\n* Employer: Direction Software Solutions, Mumbai\n* Work Mode: Permanent Work from Home\n* CTC/Packaging: ₹17 LPA\n\n### Personal Details\n\n* Date of Birth: 09 November 1996\n* Time of Birth: 11:30 AM\n* Place of Birth: Thiruttani (Ammamma Gari Illu)\n* Caste: OC (Kapu Reddy) — Pakanati\n* Gotram: Achinthala\n* Nakshatram: Chitha\n* Height: 5